# 🔄 Replay Episodes to Robot

Loops through all episodes in a LeRobot v3.0 dataset and replays them one after another.

⚠️ **Warning**: This will move the physical robot! Ensure the workspace is clear.

In [ ]:
from pathlib import Path

# ──────────────────────────────────────────────
# Option A: Local dataset
# Set DATASET_DIR to your local LeRobot v3.0 dataset path. Leave as None to use HuggingFace instead.
DATASET_DIR = None  # e.g. Path("/data/lerobot/your_dataset")

# ──────────────────────────────────────────────
# Option B: HuggingFace Hub dataset
# Set HF_REPO_ID to a dataset repo on HuggingFace. Leave as None to use local instead.
HF_REPO_ID = None  # e.g. "pokeandwiggle/my_dataset" #UPDATE-ME

# Local directory to store downloaded HuggingFace datasets
HF_DOWNLOAD_DIR = Path("/data/lerobot")

# ──────────────────────────────────────────────
# Validation
if DATASET_DIR and HF_REPO_ID:
    raise ValueError("Set only one of DATASET_DIR or HF_REPO_ID, not both.")
if not DATASET_DIR and not HF_REPO_ID:
    raise ValueError("Set either DATASET_DIR (local) or HF_REPO_ID (HuggingFace).")

# If using HuggingFace, download the dataset snapshot to a local cache path
if HF_REPO_ID:
    from huggingface_hub import snapshot_download
    # Build a subfolder name from the repo id (e.g. "org/dataset" → "org_dataset")
    local_dir = HF_DOWNLOAD_DIR / HF_REPO_ID.replace("/", "_")
    print(f"⬇️  Downloading dataset '{HF_REPO_ID}' from HuggingFace Hub...")
    DATASET_DIR = Path(snapshot_download(repo_id=HF_REPO_ID, local_dir=local_dir, repo_type="dataset"))
    print(f"✅ Downloaded to: {DATASET_DIR}")
else:
    DATASET_DIR = Path(DATASET_DIR)
    print(f"✅ Using local dataset: {DATASET_DIR}")

# ──────────────────────────────────────────────
# Robot & replay configuration
# TODO: Change to the robot's IP address.
SERVER_ENDPOINT = "<robot_ip_address>:50051"      # robot gRPC address #UPDATE-ME
MOUNT_TYPE = "wall"                               # "wall" or "table" #UPDATE-ME
REPLAY_FREQUENCY_HZ = 30.0                        # should match recording FPS #UPDATE-ME
SPEED_MULTIPLIER = 1.0                            # 1.0 = normal speed

print(f"Dataset:          {DATASET_DIR}")
print(f"Robot endpoint:   {SERVER_ENDPOINT}")
print(f"Replay frequency: {REPLAY_FREQUENCY_HZ} Hz")

In [ ]:
import time
import numpy as np
import torch
import pyarrow.parquet as pq
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

from example_policies.robot_deploy.deploy_core.policy_loader import load_metadata
from example_policies.robot_deploy.deploy_core.action_translator import ActionTranslator
from example_policies.robot_deploy.deploy import move_home
from example_policies.robot_deploy.robot_io.connection import RobotConnection
from example_policies.robot_deploy.robot_io.robot_interface import RobotInterface
from example_policies.robot_deploy.robot_io.robot_client import RobotClient
from example_policies.robot_deploy.utils import print_info
from example_policies.utils.constants import ACTION, OBSERVATION_STATE
from example_policies.utils.action_order import ActionMode

# ── Load metadata & build config ────────────────────────────
metadata = load_metadata(DATASET_DIR)
ds_meta = LeRobotDatasetMetadata(repo_id=DATASET_DIR.name, root=DATASET_DIR)
num_episodes = ds_meta.total_episodes

device = "cuda" if torch.cuda.is_available() else "cpu"

class FakeConfig:
    def __init__(self, m):
        self.metadata = m
        self.device = device
        self.output_features = {}
        self.input_features = {}
        self.input_features[OBSERVATION_STATE] = np.asarray(m["features"][OBSERVATION_STATE]["names"])
        self.output_features[ACTION] = np.asarray(m["features"][ACTION]["names"])

    def get_tcp_from_state(self, state):
        names = (
            [f"tcp_left_pos_{i}" for i in "xyz"]
            + [f"tcp_left_quat_{i}" for i in "xyzw"]
            + [f"tcp_right_pos_{i}" for i in "xyz"]
            + [f"tcp_right_quat_{i}" for i in "xyzw"]
        )
        idx = [np.where(self.input_features[OBSERVATION_STATE] == n)[0][0] for n in names]
        return state[:, idx]

cfg = FakeConfig(metadata)
action_translator = ActionTranslator(cfg)

print(f"✅ Ready — {num_episodes} episodes, action mode: {action_translator.action_mode}")

## 3. (Optional) Move Robot to Home Position

Before replaying, you can move the robot to a predefined home position with grippers open. This ensures a consistent starting pose.

In [ ]:
from example_policies.robot_deploy.deploy import move_home
from example_policies.robot_deploy.robot_io.connection import RobotConnection
from example_policies.robot_deploy.robot_io.robot_interface import RobotInterface

# Choose mount type: "table" or "wall"
MOUNT_TYPE = "wall"  # UPDATE-ME

# Move robot to home position
with RobotConnection(SERVER_ENDPOINT) as stub:
    robot_interface = RobotInterface(stub, cfg)
    move_home(robot_interface, mount=MOUNT_TYPE)

print("✅ Robot at home position, ready for replay.")

## 4. Replay Episodes

Execute the cell below to replay all episodes. The robot will return home between each episode.

⚠️ **Warning**: This will move the physical robot! Ensure the workspace is clear.

In [ ]:
def replay_episode(robot_interface, dbg_printer, ep_index: int):
    """Replay a single episode: move to start, then send all actions."""
    print(f"\n{'='*60}")
    print(f"  Episode {ep_index + 1}/{num_episodes} (index {ep_index})")
    print(f"{'='*60}")

    # Load episode actions from parquet (fast, no video decoding)
    data_file = ds_meta.get_data_file_path(ep_index)
    table = pq.read_table(DATASET_DIR / data_file)
    mask = table.column("episode_index").to_pylist()
    rows = [i for i, ep in enumerate(mask) if ep == ep_index]
    table = table.take(rows)

    actions = np.array(table.column("action").to_pylist(), dtype=np.float32)
    states = np.array(table.column("observation.state").to_pylist(), dtype=np.float32)
    print(f"  {len(actions)} frames, ~{len(actions)/REPLAY_FREQUENCY_HZ:.1f}s")

    # ── Move to start position ──────────────────────────────
    if action_translator.action_mode in (ActionMode.DELTA_TCP, ActionMode.ABS_TCP, ActionMode.TCP):
        tcp = cfg.get_tcp_from_state(states[0:1])
        gripper = np.array([1.0, 1.0])  # grippers open
        start = np.concatenate([tcp.flatten(), gripper]).astype(np.float32)[None, :]
        print(f"  Moving to start position...")
        robot_interface.send_action(
            torch.from_numpy(start), ActionMode.ABS_TCP, RobotClient.CART_WAYPOINT
        )
        time.sleep(2.0)  # give the robot time to reach start

    # ── Replay actions ──────────────────────────────────────
    period = 1.0 / (REPLAY_FREQUENCY_HZ * SPEED_MULTIPLIER)
    for step, action in enumerate(actions):
        t0 = time.time()
        obs = robot_interface.get_observation(device)
        if obs:
            action_tensor = torch.tensor(action, dtype=torch.float32).unsqueeze(0)
            translated = action_translator.translate(action_tensor, obs)
            if step % 50 == 0:
                dbg_printer.print(step, obs, translated, raw_action=False)
            robot_interface.send_action(translated, action_translator.action_mode, RobotClient.CART_WAYPOINT)
        time.sleep(max(0.0, period - (time.time() - t0)))

    print(f"  ✅ Episode {ep_index} done")


# ── Loop through all episodes ───────────────────────────────
with RobotConnection(SERVER_ENDPOINT) as stub:
    robot_interface = RobotInterface(stub, cfg)
    dbg_printer = print_info.InfoPrinter(cfg)

    for ep in range(num_episodes):
        replay_episode(robot_interface, dbg_printer, ep)
        # Return home between episodes
        move_home(robot_interface, mount=MOUNT_TYPE)
        time.sleep(1.0)

print(f"\n🏁 All {num_episodes} episodes replayed!")